# dk EGAT 모델 풀 학습 (Colab A100)

> **최종 업데이트**: 2026-07-14 (Gated Fusion + Bilinear Classifier 채택 확정, 셀 3 커맨드 변경):
> CPU A/B 스크리닝(distant 3,000문서) 결과 Gated Fusion(F1 27.38 vs baseline 25.16)과 ATLOP식
> Grouped Bilinear Classifier(F1 28.79 vs baseline 25.12) 둘 다 확실히 이겨서 **`--use_gated_fusion
> --use_bilinear_classifier` 필수 추가**. Jump Knowledge on/off, abs-diff는 각각 Gated
> Fusion/Bilinear Classifier가 그 자리를 대체해서 무의미해짐(둘 다 이겼지만 최종 조합엔 안 넣음) —
> 상세 비교표는 `Scripts/dk_gat/README.md` 참고. 추가로 학습 전략 중 **`--lr2 1e-5`**(stage2
> fine-tune만 더 낮은 LR)와 **`--early_stop_patience 3`**(best-checkpoint는 이미 별도 추적되므로
> 안전한 조기 종료)도 반영 — 나머지 학습 전략 플래그(`--layerwise_lr_decay`,
> `--freeze_encoder_epochs`, `--evidence_start_epoch`)는 이번 실행엔 보류(미검증 변수를 한 번에
> 너무 많이 섞지 않기 위함).
>
> 이전 (그래프/pair representation 고도화): ATLoss 교체 효과가 실측으로
> 확인됨(distant 프리트레인만 dev F1 46.58/Ign 44.21 — BCE 때 24.77에서 회복, 20k 기준 참고치
> 43.15~47.79 범위 안) → 보류해뒀던 고도화 2건 반영.
>
> ① **Entity-Sentence Heterogeneous Graph**: 그래프 노드에 문장(Sentence)을 추가, entity-entity
> 엣지에 더해 entity-sentence("등장함") 엣지 신설. 직접/같은 문장/멘션겹침으로 안 이어진 두
> 엔티티도 공통 문장 노드를 거쳐 2-hop 만에 정보 교환 가능 (예: Steve Jobs -[S1]- Apple -[S2]-
> California). GAT 통과 후 엔티티 행만 잘라내 pair 구성에 사용.
>
> ② **Pair Representation에 element-wise interaction 추가**: `[g_h ‖ g_t ‖ c_ht]`(2304-d)에
> `g_h*g_t`(768-d) 곱 항을 더해 `[g_h ‖ g_t ‖ g_h*g_t ‖ c_ht]`(3072-d)로 확장.
>
> ③ **best-epoch 체크포인트 저장 추가**: 매 epoch dev F1이 갱신될 때마다 `{run_name}_best.pt`로
> 저장 — 마지막 epoch이 우연히 저점일 때(실측: epoch 6 F1 59.85 → epoch 7 59.57 하락 관측) 최고
> 기록을 놓치지 않기 위함. 최종 epoch 체크포인트(`{run_name}.pt`, baseline과 비교 기준)는 별도 유지.
>
> 이전: 손실함수를 BCE+threshold sweep → Adaptive Thresholding(ATLoss/PUATLoss)으로 교체 (distant
> 단계 PUATLoss na_weight=0.7, annotated는 자동 전환). 파이프라인 상세는 `Scripts/dk_gat/README.md`.

**실행 전**: 런타임 → 런타임 유형 변경 → **A100 GPU**.

**학습 구성**: distant 20,000개 × **1 epoch** PUATLoss(na_weight=0.7) 프리트레인 → annotated
3,053개 × **최대 15 epoch**(early stopping patience=3) ATLoss 파인튜닝 (stage1 AdamW 2e-5, stage2
1e-5, wd 0.01, warmup 6%, clip 1.0, seed 66) + **Gated Fusion + Grouped Bilinear Classifier** —
**`Scripts/atlop` baseline과 distant_limit/distant_epochs/epochs/seed를 전부 동일하게 맞춰서
GAT(+Gated Fusion+Bilinear) 추가라는 아키텍처 변수를 비교**함. 예측은 Adaptive Thresholding으로
결정(전역 threshold 없음, 페어마다 학습된 TH 클래스와 비교) — 더 이상 threshold sweep 불필요.

**비교 기준** (`results/comparison.md`, 전부 동일 스코어러·seed 66·distant 20k·distant_epochs 1):

> **⚠️ 참고 (2026-07-14 뒤늦게 발견)**: 바로 위 "Heterogeneous graph + interaction term" 행은
> 사실 이미 실측 완료된 실행이었음(노트북 셀 4의 예전 출력에 그대로 남아있었는데 이번에 발견함) —
> **dev F1 60.29 / Ign F1 58.39 (best epoch 13)로 baseline(61.71/59.86)보다 낮음**. Gated
> Fusion(+2.2 F1)·Bilinear Classifier(+3.67 F1) 스크리닝 결과가 그대로 전이된다면 이 60.29를
> baseline 이상으로 끌어올릴 걸로 기대되지만, **이 프로젝트에서 이미 한 번 확인된 패턴**(PU
> loss가 distant-only 스크리닝에서 +2.93 F1이었다가 실제 파인튜닝 후엔 +0.35로 줄어든 사례,
> `results/comparison.md` 참고)을 감안하면 스크리닝 이득이 그대로 다 전이된다는 보장은 없음 —
> 이번 실행이 실제 확인 수단.


| 모델 | annotated epochs | dev F1 | Ign F1 |
|---|---|---|---|
| ATLOP baseline | 15 | 61.71 | 59.86 |
| ATLOP + PUATLoss(0.7) | 15 | 62.06 | 60.16 |
| dk EGAT (엔티티 그래프, ATLoss 교체 직후) | 8/15까지 관측 | 59.85(peak) | 58.12(peak) |
| dk EGAT + Heterogeneous graph + interaction term (JK/Gated Fusion/Bilinear 없음, 실측 완료) | 15 | 60.29(peak, epoch13) | 58.39(peak) |
| **dk EGAT + Gated Fusion + Bilinear Classifier (이 실행)** | 15(early stop patience 3) | | |


In [1]:
# 0) GPU 확인 (A100이 보여야 함)
!nvidia-smi -L

GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-368661c4-01eb-742c-550a-6a186b616bfa)


In [2]:
# 1) 코드 + 데이터 (Git LFS에서 필요한 json만 선별 pull)
#    Scripts/dk_gat는 아직 main이 아니라 dk 브랜치에만 있으므로 반드시 checkout 필요
!GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/multiful/Information_Extraction.git
%cd Information_Extraction
!git checkout dk
!git lfs pull --include="docred_data/data/dev.json,docred_data/data/train_annotated.json,docred_data/data/train_distant.json,docred_data/data/rel_info.json"
# json들이 MB~GB 단위로 보이면 성공 (133바이트면 LFS pull 실패)
!ls -lh docred_data/data/*.json
# Scripts/dk_gat가 실제로 있는지 확인 (여기 안 뜨면 아래 학습 셀에서 ModuleNotFoundError 남)
!ls Scripts/dk_gat/

Cloning into 'Information_Extraction'...
remote: Enumerating objects: 365, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 365 (delta 63), reused 84 (delta 40), pack-reused 241 (from 1)
Receiving objects: 100% (365/365), 22.18 MiB | 17.87 MiB/s, done.
Resolving deltas: 100% (171/171), done.
/content/Information_Extraction
Branch 'dk' set up to track remote branch 'dk' from 'origin'.
Switched to a new branch 'dk'
-rw-r--r-- 1 root root 4.1M Jul 14 08:16 docred_data/data/dev.json
-rw-r--r-- 1 root root 2.4K Jul 14 08:16 docred_data/data/rel_info.json
-rw-r--r-- 1 root root  132 Jul 14 08:16 docred_data/data/test.json
-rw-r--r-- 1 root root  13M Jul 14 08:16 docred_data/data/train_annotated.json
-rw-r--r-- 1 root root 417M Jul 14 08:19 docred_data/data/train_distant.json
colab_gat_a100.ipynb  model.py		 README.md
__init__.py	      preprocess_gat.py  train_gat.py


In [ ]:
# 2) 패키지
!pip install -q transformers==4.57.6 accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.1 MB/s eta 0:00:00


In [ ]:
# 3) 학습: distant 20k×1ep -> annotated 15ep (baseline과 동일 스케줄, A100 약 2~3시간)
#    epoch마다 [스테이지 | epoch N] dev_F1=.. Ign_F1=.. 로그가 찍힘
#    --use_pu_loss --na_weight 0.7: distant 단계만 PUATLoss, annotated는 자동으로 일반 ATLoss
#    (BCE+threshold sweep은 폐기 -- 실측 dev F1 24.77로 같은 20k 기준 RoBERTa+ATLoss의
#    43.15보다도 낮아서 Adaptive Thresholding으로 교체함, model.py 모듈 docstring 참고)
#    --use_gated_fusion --use_bilinear_classifier: CPU A/B 스크리닝(distant 3천 문서)으로
#    검증 완료된 채택 조합 (Gated Fusion F1 27.38 vs baseline 25.16, Bilinear Classifier
#    F1 28.79 vs baseline 25.12 -- 둘 다 필수, JK/abs-diff는 이 둘에 의해 무의미해져서 제외,
#    README.md 참고). --lr2/--early_stop_patience는 CPU 스크리닝으로 검증 불가능해 미검증
#    상태지만, early stopping은 best-checkpoint를 이미 별도 추적하므로 위험이 낮고 lr2는
#    표준적인 fine-tune LR 낮추기라 이번 실행에 포함하기로 결정함.
!python -m Scripts.dk_gat.train_gat \
  --distant_limit 20000 --distant_epochs 1 --epochs 15 \
  --use_pu_loss --na_weight 0.7 \
  --use_gated_fusion --use_bilinear_classifier \
  --lr 2e-5 --lr2 1e-5 --early_stop_patience 3 \
  --weight_decay 0.01 --warmup_ratio 0.06 --dropout 0.1 \
  --run_name dk_gat --save_model --seed 66

In [23]:
# 4) 결과 백업 (세션 끊기면 파일이 사라지므로 반드시 실행)
#    best_* = dev F1 최고 epoch 체크포인트/예측 (마지막 epoch보다 나을 수 있음, 둘 다 백업)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1"
!cp results/dk_gat.pt results/dk_gat_best.pt \
   results/dk_gat_stage1.pt results/dk_gat_stage1_best.pt \
   results/dk_gat_dev_predictions.json results/dk_gat_best_dev_predictions.json \
   "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/"
!ls -lh "/content/drive/MyDrive/MS_AI_NLP(2026)_실습자료/21_실전프로젝트1/" | grep dk_gat

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 773K Jul 14 05:49 dk_gat_best_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_best.pt
-rw------- 1 root root 759K Jul 14 05:49 dk_gat_dev_predictions.json
-rw------- 1 root root 430M Jul 14 05:49 dk_gat.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1_best.pt
-rw------- 1 root root 430M Jul 14 05:49 dk_gat_stage1.pt


## 결과 기록

마지막 epoch 로그 수치를 위의 비교표와 `results/comparison.md`에 반영. 로그에 `<- new best`가
찍힌 마지막 줄이 `dk_gat_best.pt`에 저장된 epoch — 최종 epoch 수치와 다르면 (더 낮으면) best
쪽 수치도 같이 기록해둘 것.

- stage-1(distant 직후) 수치도 따로 적어두면 GAT가 노이즈 라벨 위에서 얼마나 버티는지,
  annotated 파인튜닝이 얼마나 끌어올리는지 분해해 보여줄 수 있음.
- seed 66 단일 시드 — baseline과의 차이가 ±1점 이내면 PRD §5 시드 노이즈 주의 적용.
- Evidence Contrastive Loss는 annotated 단계에서만 실질 작동함(distant는 evidence 없음).
- GAT 고도화 추가 실험(edge feature ablation, 헤드 수, 인접행렬 반경 등)은 이 노트북의 셀 3에
  `--gat_heads`, `--evidence_weight` 인자만 바꿔 반복하면 됨.
- Meta-path별 attention 파라미터 분리, Curriculum PU-weight 스케줄은 이번에도 미반영 —
  이번 결과 확인 후 필요성 판단해 별도 실험으로 검토.